# Evaluate ai.analyze_sentiment(...) Quality with PySpark

This notebook evaluates sentiment labels with a larger judge model that supplies reference labels. The AI transformations and LLM-as-a-Judge evaluation remain in Spark. Only the small, materialized result set is converted to pandas for metrics and charts.

### What You'll Do
1. Analyze sentiment for sample text with `ai.analyze_sentiment`.
2. Try emotion, intensity, and customer-service label schemes.
3. Use `gpt-5.1` as a fixed judge for expected labels and rationales.
4. Calculate accuracy, macro precision, recall, and F1.
5. Compare the baseline with an explainable custom sentiment prompt.

### Before You Start
- **Runtime** - This notebook was made for **Fabric 1.3 runtime**.
- **Customize it** - Replace the sample data and adapt the judge criteria to your use case.
- **Keep comparisons fair** - Hold the judge model and prompts fixed while changing one executor setting at a time.
- **Validate important decisions** - LLM judge scores are useful proxies, not a substitute for human-reviewed production samples.

| Metric | Measures |
|--------|----------|
| **Accuracy** | Overall label correctness |
| **Precision** | Correct predictions within each predicted class |
| **Recall** | Coverage of each expected class |
| **F1 score** | Balance of precision and recall |

[ai.analyze_sentiment PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/analyze-sentiment)


## 1. Setup

Install Pydantic for the structured response schemas used by the judge.
The executor uses `gpt-5-mini` with low reasoning effort. The judge uses
`gpt-5.1` with medium reasoning effort and remains fixed across comparisons.
See the [AI Functions model and CU rate table](https://aka.ms/aifunctions-fabric-llm-cu-rates)
for all available models.


In [ ]:
%pip install -q pydantic 2>/dev/null


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from pydantic import BaseModel, Field
from pyspark.sql import functions as F

# Use the smaller model for the function under test and a larger fixed model for judging.
EXECUTOR_OPTIONS = {
    "deploymentName": "gpt-5-mini",
    "reasoningEffort": "low",
}
JUDGE_OPTIONS = {
    "deploymentName": "gpt-5.1",
    "reasoningEffort": "medium",
}

def materialize(frame):
    cached = frame.cache()
    _ = cached.count()
    error_columns = [name for name in cached.columns if name.endswith("_error")]
    if error_columns:
        has_error = F.lit(False)
        for name in error_columns:
            has_error = has_error | F.coalesce(
                F.length(F.trim(F.col(name).cast("string"))) > 0,
                F.lit(False),
            )
        failed_rows = cached.filter(has_error)
        failure_count = failed_rows.count()
        if failure_count:
            print(f"{failure_count} row(s) contain AI Function errors:")
            id_columns = [
                name for name in ("sample_id", "ticket_id") if name in cached.columns
            ]
            display(failed_rows.select(*id_columns, *error_columns))
    return cached

def fresh_ai_view(frame):
    return frame.select("*")


## 2. Load Sample Data


In [ ]:
rows = [(1,
  "I've used this laptop for a month and it's been great. The screen is sharp, the keyboard feels good, and "
  'the battery lasts a full workday.'),
 (2,
  'My order arrived over a week late, and the device was cracked when I opened the box. Support kept me on '
  "hold and never followed up, so I'm very disappointed."),
 (3,
  'The strategy meeting had strong market insights, but the budget section ended without decisions. I left '
  'with useful notes and unresolved action items.'),
 (4,
  'Dinner at the new Thai restaurant was delicious, especially the noodles and dessert. Service was slower '
  'than expected, but the staff were polite and helpful.'),
 (5,
  'This project tool helps us plan sprints, but the mobile app crashes often and reports are hard to use. It '
  'saves time in some areas and creates extra work in others.'),
 (6,
  'The city council approved the 2025 budget and published department allocations. Public works funding '
  'increased, while parks funding stayed the same.'),
 (7,
  'The revised proposal moved the deadline up by two weeks even after the team raised capacity concerns. I '
  'am worried this timeline will lead to rushed work.'),
 (8,
  'I came back from the data engineering summit energized. The keynote was practical, and I left with '
  'concrete ideas to try with my team.')]
df = spark.createDataFrame(rows, ["sample_id", "text"])
display(df)


## 3. Run `ai.analyze_sentiment`

Spark AI transformations are lazy. Cache and materialize reused results once so
later displays, judging, and metrics do not repeat model calls.


In [ ]:
sentiment_df = materialize(
    df.ai.analyze_sentiment(
        input_col="text",
        output_col="sentiment",
        error_col="executor_error",
        **EXECUTOR_OPTIONS,
    )
)
display(sentiment_df.select("text", "sentiment"))
display(sentiment_df.ai.stats)


### 3.1 Optional: Custom Sentiment Labels

Domain-specific labels can capture emotion, intensity, or customer-service
urgency that the default four-label scheme does not express.


In [ ]:
CUSTOM_LABEL_SCHEMES = {
    "emotion": ["excited", "satisfied", "disappointed", "angry", "confused"],
    "intensity": [
        "very_positive",
        "positive",
        "neutral",
        "negative",
        "very_negative",
    ],
    "urgency": [
        "urgent_complaint",
        "mild_concern",
        "neutral_inquiry",
        "positive_feedback",
        "enthusiastic_praise",
    ],
}

custom_labels_df = fresh_ai_view(sentiment_df)
for output_col, labels in CUSTOM_LABEL_SCHEMES.items():
    custom_labels_df = fresh_ai_view(custom_labels_df).ai.analyze_sentiment(
        labels=labels,
        input_col="text",
        output_col=output_col,
        error_col=f"{output_col}_error",
        **EXECUTOR_OPTIONS,
    )
custom_labels_df = materialize(custom_labels_df)
display(
    custom_labels_df.select(
        "text", "sentiment", *CUSTOM_LABEL_SCHEMES.keys()
    )
)


## 4. Evaluate with an LLM Judge


In [ ]:
from typing import Literal

class SentimentEval(BaseModel):
    reason: str = Field(description="Brief rationale for the expected label")
    expected_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="Best sentiment label for the text"
    )

EVAL_PROMPT = """You are evaluating a sentiment prediction.

Labels:
- positive: satisfaction, happiness, praise, or approval
- negative: dissatisfaction, frustration, criticism, or disapproval
- neutral: factual or objective language without a clear emotional tone
- mixed: meaningful positive and negative sentiment in the same text

<text>
{text}
</text>
<predicted_sentiment>
{sentiment}
</predicted_sentiment>

Return a brief rationale and the best expected label."""


In [ ]:
evaluated_df = fresh_ai_view(sentiment_df).ai.generate_response(
    prompt=EVAL_PROMPT,
    is_prompt_template=True,
    output_col="_eval_response",
    error_col="_eval_error",
    response_format=SentimentEval,
    **JUDGE_OPTIONS,
)
evaluated_df = (
    evaluated_df
    .withColumn(
        "expected_sentiment",
        F.get_json_object(F.col("_eval_response"), "$.expected_sentiment"),
    )
    .withColumn(
        "correct",
        F.col("sentiment") == F.col("expected_sentiment"),
    )
    .withColumn(
        "eval_reason",
        F.get_json_object(F.col("_eval_response"), "$.reason"),
    )
)
evaluated_df = materialize(evaluated_df)
display(
    evaluated_df.select(
        "text", "sentiment", "expected_sentiment", "correct", "eval_reason"
    )
)


## 5. Results


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

results_pd = evaluated_df.select(
    "sample_id", "text", "sentiment", "expected_sentiment", "correct", "eval_reason"
).toPandas()
metric_input = results_pd.dropna(subset=["expected_sentiment", "sentiment"]).copy()
excluded_rows = len(results_pd) - len(metric_input)
if excluded_rows:
    print(f"Excluded {excluded_rows} row(s) without both expected and predicted labels.")
if metric_input.empty:
    raise ValueError("No valid sentiment rows are available for metric calculation.")
y_true = metric_input["expected_sentiment"]
y_pred = metric_input["sentiment"]

metric_names = ["Accuracy", "Precision", "Recall", "F1 score"]
metric_values = [
    accuracy_score(y_true, y_pred),
    precision_score(y_true, y_pred, average="macro", zero_division=0),
    recall_score(y_true, y_pred, average="macro", zero_division=0),
    f1_score(y_true, y_pred, average="macro", zero_division=0),
]
metrics_pd = pd.DataFrame({"Metric": metric_names, "Score": metric_values})
display(metrics_pd.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars = axes[0].bar(
    metric_names,
    metric_values,
    color=["#0077aa", "#22cc77", "#9955bb", "#ee7722"],
)
axes[0].set_ylim(0, 1)
axes[0].set_title("Sentiment Classification Metrics")
axes[0].axhline(y=0.9, color="#999999", linestyle="--", alpha=0.6)
axes[0].bar_label(bars, fmt="%.2f", padding=2)

correct_count = int((y_true == y_pred).sum())
axes[1].pie(
    [correct_count, len(metric_input) - correct_count],
    labels=["Correct", "Incorrect"],
    autopct="%1.0f%%",
    colors=["#22cc77", "#ee4433"],
)
axes[1].set_title("Judge-Derived Accuracy")
plt.tight_layout()
plt.show()

report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
labels = sorted(set(y_true) | set(y_pred))
per_class_pd = pd.DataFrame([
    {
        "sentiment": label,
        "precision": report[label]["precision"],
        "recall": report[label]["recall"],
        "f1_score": report[label]["f1-score"],
        "support": int(report[label]["support"]),
    }
    for label in labels
    if label in report
])
display(per_class_pd.round(3))


In [ ]:
breakdown_pd = results_pd[
    ["text", "sentiment", "expected_sentiment", "correct", "eval_reason"]
].copy()
breakdown_pd["status"] = breakdown_pd["correct"].apply(
    lambda value: "PASS" if value == True else "REVIEW"
)
breakdown_pd["text"] = breakdown_pd["text"].str[:120] + "..."
display(breakdown_pd)


## 6. Optional Refinement: Explainable Custom Sentiment

Reuse the judge-derived expected labels to compare the baseline with a custom
structured prompt that returns both a label and a short rationale.


In [ ]:
class CustomSentimentPrediction(BaseModel):
    reason: str = Field(description="Brief rationale for the label")
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="Best sentiment label"
    )

CUSTOM_SENTIMENT_PROMPT = """Classify the sentiment of the text as positive, negative,
neutral, or mixed. Return a brief rationale and exactly one label.

<text>
{text}
</text>"""

custom_df = fresh_ai_view(evaluated_df).ai.generate_response(
    prompt=CUSTOM_SENTIMENT_PROMPT,
    is_prompt_template=True,
    output_col="_custom_response",
    error_col="_custom_error",
    response_format=CustomSentimentPrediction,
    **EXECUTOR_OPTIONS,
)
custom_df = (
    custom_df
    .withColumn(
        "custom_sentiment",
        F.get_json_object(F.col("_custom_response"), "$.sentiment"),
    )
    .withColumn(
        "custom_reason",
        F.get_json_object(F.col("_custom_response"), "$.reason"),
    )
)
custom_df = materialize(custom_df)
display(
    custom_df.select(
        "text", "sentiment", "expected_sentiment", "custom_sentiment", "custom_reason"
    )
)


In [ ]:
custom_pd = custom_df.select(
    "sample_id", "expected_sentiment", "sentiment", "custom_sentiment"
).toPandas()
required_columns = ["expected_sentiment", "sentiment", "custom_sentiment"]
paired_mask = custom_pd[required_columns].notna().all(axis=1)
paired_count = int(paired_mask.sum())
excluded_count = int((~paired_mask).sum())
print(f"Paired rows: {paired_count} | Excluded rows: {excluded_count}")
if excluded_count:
    display(custom_pd.loc[~paired_mask, ["sample_id", *required_columns]])
if not paired_count:
    raise ValueError("No rows have both baseline and custom sentiment results.")

paired_pd = custom_pd.loc[paired_mask]
expected = paired_pd["expected_sentiment"]

def classification_metrics(column_name):
    predictions = paired_pd[column_name]
    return [
        accuracy_score(expected, predictions),
        precision_score(expected, predictions, average="macro", zero_division=0),
        recall_score(expected, predictions, average="macro", zero_division=0),
        f1_score(expected, predictions, average="macro", zero_division=0),
    ]

comparison_pd = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 score"],
    "Baseline": classification_metrics("sentiment"),
    "Custom": classification_metrics("custom_sentiment"),
})
comparison_pd["Delta"] = comparison_pd["Custom"] - comparison_pd["Baseline"]
display(comparison_pd.round(3))

ax = comparison_pd.set_index("Metric")[["Baseline", "Custom"]].plot.bar(
    figsize=(8, 4),
    color=["#0077aa", "#22cc77"],
    rot=0,
)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Baseline vs Custom Sentiment")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2)
plt.tight_layout()
plt.show()


## Interpreting Results

| Metric score | Suggested action |
|--------------|------------------|
| **0.80-1.00** | Strong result; review per-class metrics and flagged samples |
| **0.70-0.79** | Good starting point; investigate weaker classes |
| **Below 0.70** | Refine labels, data coverage, or the executor approach |

| Metric or issue | Likely cause | Next step |
|-----------------|--------------|-----------|
| Low accuracy | Ambiguous, sarcastic, or domain-specific text | Refine labels and add reviewed examples |
| Low precision | A label is over-predicted | Clarify boundaries between labels |
| Low recall | A label is under-detected | Add representative samples for that class |

### Improving Quality

- Use domain-specific sentiment labels when the default four labels are too broad.
- For harder cases, test the executor with `deploymentName="gpt-5.1"` and
  `reasoningEffort="medium"`, then compare it with the same judge.
- Add human-reviewed examples for sarcasm, short text, and mixed sentiment.

Keep the judge configuration fixed for comparisons, and confirm release decisions with representative human-reviewed samples.

## Learn More

- [ai.analyze_sentiment PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/analyze-sentiment)
- [AI Functions overview](https://learn.microsoft.com/fabric/data-science/ai-functions/overview)
